# Allerion — Live API Test Notebook
Manual verification of every endpoint on the deployed Cloud Run service.

In [1]:
import httpx
import json

BASE = "https://allerion-yho3hnnula-uc.a.run.app"

def show(resp: httpx.Response):
    print(f"HTTP {resp.status_code}")
    try:
        print(json.dumps(resp.json(), indent=2))
    except Exception:
        print(resp.text)

## 1. Species list
`GET /api/species` — returns all 15 tracked allergen species.

In [2]:
resp = httpx.get(f"{BASE}/api/species", timeout=30)
show(resp)

HTTP 200
[
  {
    "species_id": 53587,
    "name": "Common ragweed",
    "pollen_type": "weed",
    "allergenicity": 5
  },
  {
    "species_id": 54779,
    "name": "White oak",
    "pollen_type": "tree",
    "allergenicity": 4
  },
  {
    "species_id": 49883,
    "name": "Paper birch",
    "pollen_type": "tree",
    "allergenicity": 4
  },
  {
    "species_id": 49399,
    "name": "Eastern red cedar",
    "pollen_type": "tree",
    "allergenicity": 3
  },
  {
    "species_id": 57190,
    "name": "Timothy grass",
    "pollen_type": "grass",
    "allergenicity": 4
  },
  {
    "species_id": 54808,
    "name": "Green ash",
    "pollen_type": "tree",
    "allergenicity": 3
  },
  {
    "species_id": 53547,
    "name": "American elm",
    "pollen_type": "tree",
    "allergenicity": 3
  },
  {
    "species_id": 52119,
    "name": "Eastern cottonwood",
    "pollen_type": "tree",
    "allergenicity": 2
  },
  {
    "species_id": 57140,
    "name": "Olive",
    "pollen_type": "tree",
    "all

## 2. Forecast — 14-day species-level
`GET /api/forecast?lat=&lng=` — change lat/lng to your demo location.

In [3]:
resp = httpx.get(f"{BASE}/api/forecast", params={"lat": 32.7, "lng": -117.1}, timeout=60)
show(resp)

HTTP 200
{
  "location": {
    "lat": 32.7,
    "lng": -117.1,
    "city": "",
    "h3_cell": "8429a41ffffffff"
  },
  "generated_at": "2026-04-19T10:33:06.250369",
  "daily": [
    {
      "date": "2026-04-19",
      "day_offset": 0,
      "confidence_tier": "high",
      "composite_index": 1.77,
      "severity": "moderate",
      "top_species": [
        {
          "species_id": 60307,
          "name": "Kentucky bluegrass",
          "pollen_type": "grass",
          "current_stage": "EARLY_BLOOM",
          "pollen_prob": 0.483,
          "pollen_index": 1.84,
          "days_to_peak": 23,
          "peak_date_est": "2026-05-12",
          "confidence": 0.9,
          "sources": [
            "base",
            "google"
          ],
          "seasonal_shift_days": 0,
          "inat_obs_count": 0,
          "google_upi": 3
        },
        {
          "species_id": 52801,
          "name": "Perennial ryegrass",
          "pollen_type": "grass",
          "current_stage": "EAR

In [4]:
# Quick summary — today only
data = resp.json()
today = data["daily"][0]
print("Date:     ", today["date"])
print("Severity: ", today["severity"])
print("Composite:", today["composite_index"])
print("Headline: ", data["narrative"]["headline"])
print()
print("Top species today:")
for sp in today["top_species"]:
    print(f"  {sp['name']:30s}  index={sp['pollen_index']:.2f}  stage={sp['current_stage']}")

Date:      2026-04-19
Severity:  moderate
Composite: 1.77
Headline:  Moderate pollen levels today, driven by grass and olive.

Top species today:
  Kentucky bluegrass              index=1.84  stage=EARLY_BLOOM
  Perennial ryegrass              index=1.81  stage=EARLY_BLOOM
  Olive                           index=1.67  stage=EARLY_BLOOM
  Timothy grass                   index=1.30  stage=BUDDING
  Northern red oak                index=0.87  stage=POST_BLOOM


## 3. Heatmap
`GET /api/heatmap?lat=&lng=` — GeoJSON of H3 cells for the bounding area.

In [5]:
resp = httpx.get(f"{BASE}/api/heatmap", params={"lat": 32.7, "lng": -117.1}, timeout=60)
data = resp.json()
print(f"HTTP {resp.status_code}")
print(f"Type: {data.get('type')}")
print(f"Features: {len(data.get('features', []))}")
if data.get('features'):
    f = data['features'][0]
    print("First cell properties:", f['properties'])

HTTP 200
Type: FeatureCollection
Features: 7
First cell properties: {'h3_cell': '8429a41ffffffff', 'lat': 32.8262, 'lng': -117.2357, 'composite_index': 1.18, 'severity': 'low', 'top_species_name': 'Olive', 'top_species_prob': 0.657}


## 4. User profile — create / read / update / delete

In [6]:
# CREATE  (or update if user_id already exists)
profile = {
    "user_id": "demo-user-1",
    "trigger_species": [53587, 49883],   # ragweed + birch
    "severity_threshold": "low",          # 'low' always triggers in demo
    "lat": 32.7,
    "lng": -117.1,
    "fcm_token": ""                       # paste real FCM token here for push notifications
}
resp = httpx.post(f"{BASE}/api/profile", json=profile, timeout=30)
show(resp)

HTTP 200
{
  "user_id": "demo-user-1",
  "trigger_species": [
    53587,
    49883
  ],
  "severity_threshold": "low",
  "lat": 32.7,
  "lng": -117.1,
  "fcm_token": "",
  "created_at": "2026-04-19T11:21:30.523791",
  "updated_at": "2026-04-19T11:21:30.523843"
}


In [7]:
# READ
resp = httpx.get(f"{BASE}/api/profile/demo-user-1", timeout=30)
show(resp)

HTTP 200
{
  "user_id": "demo-user-1",
  "trigger_species": [
    53587,
    49883
  ],
  "severity_threshold": "low",
  "lat": 32.7,
  "lng": -117.1,
  "fcm_token": "",
  "created_at": "2026-04-19T11:21:30.523791",
  "updated_at": "2026-04-19T11:21:30.523843"
}


In [8]:
# UPDATE — raise threshold
profile["severity_threshold"] = "moderate"
resp = httpx.post(f"{BASE}/api/profile", json=profile, timeout=30)
data = resp.json()
print(f"HTTP {resp.status_code}")
print("updated_at:", data["updated_at"])
print("threshold: ", data["severity_threshold"])

HTTP 200
updated_at: 2026-04-19T11:21:31.755502
threshold:  moderate


## 5. Alert check
`POST /api/alerts/check` — evaluates all stored profiles and sends FCM push notifications to devices with a real `fcm_token`.

In [9]:
resp = httpx.post(f"{BASE}/api/alerts/check", timeout=60)
show(resp)

HTTP 200
{
  "alerts_evaluated": 1,
  "alerts_sent": 0,
  "details": []
}


In [10]:
# Interpret the result
data = resp.json()
print(f"Profiles evaluated : {data['alerts_evaluated']}")
print(f"Notifications sent : {data['alerts_sent']}")
for d in data["details"]:
    sent = "✓ sent" if d["notification_sent"] else "✗ skipped (no FCM token or below threshold)"
    print(f"  {d['user_id']:20s}  {d['species_name']:25s}  {d['severity']:12s}  {sent}")

Profiles evaluated : 1
Notifications sent : 0


## 6. Photo classifier
`POST /api/photo` — upload a plant photo, returns species identification + allergenicity.

In [ ]:
# Replace with a real image path
IMAGE_PATH = "/path/to/plant_photo.jpg"

with open(IMAGE_PATH, "rb") as f:
    resp = httpx.post(
        f"{BASE}/api/photo",
        files={"file": ("photo.jpg", f, "image/jpeg")},
        timeout=60,
    )
show(resp)

## 7. iNat delta ingest
`POST /api/ingest/delta` — trigger a manual iNaturalist observation pull for a bounding box.

In [ ]:
body = {
    "regions": [
        {
            "name": "san_diego",
            "swlat": 32.5, "swlng": -117.3,
            "nelat": 33.1, "nelng": -116.8
        }
    ]
}
resp = httpx.post(f"{BASE}/api/ingest/delta", json=body, timeout=120)
show(resp)

## 8. Cleanup — delete test profile

In [ ]:
resp = httpx.delete(f"{BASE}/api/profile/demo-user-1", timeout=30)
show(resp)

## 9. OpenAPI / Swagger UI
Open in browser for interactive exploration.

In [ ]:
print(f"Swagger UI : {BASE}/docs")
print(f"ReDoc      : {BASE}/redoc")
print(f"OpenAPI JSON: {BASE}/openapi.json")

# Verify schema is valid
resp = httpx.get(f"{BASE}/openapi.json", timeout=30)
schema = resp.json()
print(f"\nRoutes in schema: {list(schema['paths'].keys())}")